# [센서 실습 15] 수질 탁도 센서 – DFRobot SEN0189

탁도 센서 **SEN0189**의 출력 전압을 16비트 ADC(**ADS1115**)로 정밀하게 읽고, 제조사 특성식으로 **NTU**를 계산하여 수조의 오염을 감지합니다.

**이 노트북의 흐름**
1. 센서의 원리와 부품 사양
2. 라즈베리파이 연결 방법
3. 실제 센서를 읽는 코드 (라즈베리파이용)
4. 장비가 없을 때 쓰는 **가상 센서** (같은 코드로 실행)
5. 수집 → 판정 → 시각화 → 저장·IoT 전송

> 장비가 없어도 `USE_REAL_SENSOR = False` 상태 그대로 위에서부터 `Shift + Enter`로 모두 실행됩니다.

## 1. 센서의 원리
탁도는 물속에 떠 있는 입자 때문에 **물이 흐린 정도**이며 단위는 **NTU**입니다.
- 센서 안의 LED가 빛을 쏘고, 반대쪽 수광 소자가 **통과한 빛**의 양을 측정
- 물이 흐릴수록 빛이 입자에 막혀 **출력 전압이 낮아짐**
- 제조사 특성식 (5V 전원, 2.5~4.2V 구간)

$$\text{NTU} = -1120.4\,V^2 + 5742.3\,V - 4352.9$$

- 맑은 물 근처에서는 전압이 아주 조금만 변함 → 10비트 ADC(MCP3008)로는 부족해 **16비트 ADC(ADS1115)** 사용
- 기준: 수돗물 0.5 NTU 이하, 하천은 비가 오면 수십~수백 NTU

### 부품 사양
| 항목 | 내용 |
|---|---|
| 측정 범위 | 0 ~ 3,000 NTU |
| 출력 | 아날로그 0 ~ 4.5V |
| 전원 | 5V |
| 응답 시간 | 500ms 이하 |

## 2. 라즈베리파이 연결 방법
**준비물**: SEN0189 탁도 센서, ADS1115 16비트 ADC 모듈, 저항 10kΩ·20kΩ, 점퍼선

| 연결할 곳 | 라즈베리파이 | 설명 |
|---|---|---|
| ADS1115 VDD | 3.3V (1번 핀) |  |
| ADS1115 GND | GND (6번 핀) |  |
| ADS1115 SCL | GPIO3 / SCL (5번 핀) | I2C 클록 |
| ADS1115 SDA | GPIO2 / SDA (3번 핀) | I2C 데이터 |
| ADS1115 ADDR | GND | I2C 주소 0x48 |
| SEN0189 VCC / GND | 5V / GND |  |
| SEN0189 출력 → 10kΩ → ADS1115 A0 | A0 ↔ GND 사이 20kΩ | 최대 4.5V → 3.0V |

> 💡 ADS1115는 I2C로 연결되는 **16비트 ADC**입니다. 0~4.096V 범위(GAIN=1)에서 1단계가 0.125mV로, MCP3008(약 3.2mV)보다 25배 정밀합니다.

**인터페이스 활성화**
- **I2C**: `sudo raspi-config` → Interface Options → **I2C** → Enable, 연결 확인: `i2cdetect -y 1`

**라이브러리 설치 (라즈베리파이 터미널)**
```bash
pip3 install adafruit-circuitpython-ads1x15
```

## 3. 공통 준비 – 한글 폰트와 라이브러리

In [ ]:
# Colab에서 그래프의 한글이 깨지지 않도록 나눔고딕 폰트를 설정합니다. (처음 한 번 실행)
import os, subprocess
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

font_path = "/usr/share/fonts/truetype/nanum/NanumGothic.ttf"
if not os.path.exists(font_path):
    subprocess.run("apt-get -qq -y install fonts-nanum", shell=True,
                   stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    plt.rc("font", family="NanumGothic")
plt.rcParams["axes.unicode_minus"] = False
print("한글 폰트 설정 완료")

In [ ]:
import time, json
from datetime import datetime
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 4. 실제 센서 읽기 (라즈베리파이용)
### 4-1. 원시값 → 물리량 변환식 (실제·가상 공통)
ADC 전압을 센서 출력 전압으로 되돌린(×1.5) 뒤 특성식으로 NTU를 계산합니다.

In [ ]:
DIVIDER = 1.5

def voltage_to_ntu(v_sensor):
    """SEN0189 특성식 (2.5~4.2V 구간에서 유효)"""
    v = min(max(v_sensor, 2.5), 4.2)
    return round(max(0.0, -1120.4 * v ** 2 + 5742.3 * v - 4352.9), 1)

for v in (4.20, 4.15, 4.00, 3.50, 2.50):
    print(f"{v:.2f}V → {voltage_to_ntu(v):7.1f} NTU")

### 4-2. 실제 센서 클래스
Adafruit ADS1x15 라이브러리로 A0 채널 전압을 읽습니다.

> 이 셀은 **클래스를 정의만** 합니다. 하드웨어 라이브러리는 객체를 만들 때(`__init__`) 불러오므로 Colab에서 실행해도 오류가 나지 않습니다. 실제 동작은 라즈베리파이에서 `USE_REAL_SENSOR = True`로 해야 합니다.

In [ ]:
class RealSEN0189:
    def __init__(self):
        import board
        import adafruit_ads1x15.ads1115 as ADS
        from adafruit_ads1x15.analog_in import AnalogIn
        ads = ADS.ADS1115(board.I2C()); ads.gain = 1               # ±4.096V
        self.ch = AnalogIn(ads, ADS.P0)

    def read(self):
        v_sensor = self.ch.voltage * DIVIDER
        return {"v_sensor": round(v_sensor, 4), "ntu": voltage_to_ntu(v_sensor)}

## 5. 장비가 없을 때 – 가상 센서
### 왜 가상 센서를 쓰나요?
실제 센서의 값은 다음 순서로 만들어집니다.

```
① 전기 신호 (전압, 펄스, 0/1)  →  ② 원시값 raw (ADC 숫자, 바이트, 문자열)  →  ③ 변환식  →  ④ 물리량 (°C, ppm, cm …)
```
가상 센서는 ①을 건너뛰고 **②원시값을 실제 센서와 똑같은 형식으로** 만들어 줍니다.
따라서 ③변환식과 그 뒤의 판정·시각화·저장 코드는 실제 장비와 가상 장비에서 **완전히 같은 코드**를 사용합니다.

```
[장비 있음]  센서 ──배선──▶ 라즈베리파이 ──▶ Real….read()  ─┐
                                                             ├──▶ 변환식 → 판정 → 그래프 → 저장/전송
[장비 없음]  가정한 상황(시나리오) ──▶ Virtual….read() ──────┘
```
나중에 장비를 연결하면 `USE_REAL_SENSOR = True` **한 줄만** 바꾸면 됩니다.

### 가상 값은 이렇게 정했습니다
| 항목 | 가정한 값 | 근거 |
|---|---|---|
| 상황 | 양어장(수조)에 들어오는 원수, 24시간 | 수질 감시 |
| 평상시 | 약 2 NTU | 맑은 지하수·정수 |
| 오염 유입 | 8시에 비로 흙탕물 유입, 최대 약 75 NTU | 집중 호우 시 하천 원수 |
| 회복 | 침전되며 지수적으로 감소 (반감 약 2시간) | 입자 가라앉음 |
| 전압 잡음 | ±0.12 mV (10회 평균 후) | ADS1115 분해능 수준 |

가상 센서는 NTU로부터 **특성식을 거꾸로 풀어 전압**을 만들고(2차 방정식의 근), 실제와 같은 변환식으로 다시 NTU를 구합니다.

In [ ]:
class VirtualSEN0189:
    def __init__(self, dt=600, seed=15):
        self.dt, self.t = dt, 0.0
        self.rng = np.random.default_rng(seed)

    @staticmethod
    def ntu_to_voltage(ntu):
        """-1120.4 V² + 5742.3 V - (4352.9 + NTU) = 0 의 큰 근"""
        a, b, c = -1120.4, 5742.3, -(4352.9 + ntu)
        return (-b - np.sqrt(b * b - 4 * a * c)) / (2 * a)

    def read(self):
        h = self.t / 3600
        ntu = 2.0
        if h >= 8:
            ntu += 120 * (1 - np.exp(-(h - 8) / 0.5)) * np.exp(-(h - 8) / 2.9)
        v = self.ntu_to_voltage(ntu) + self.rng.normal(0, 0.00012)
        v_adc = round(v / DIVIDER, 5)                        # ADS1115가 읽는 전압
        self.t += self.dt
        return {"v_sensor": round(v_adc * DIVIDER, 4), "ntu": voltage_to_ntu(v_adc * DIVIDER)}

In [ ]:
# 가상 센서 동작 확인 – read()를 3번 호출
test = VirtualSEN0189(dt=600)
for _ in range(3):
    print(test.read())

## 6. 센서 선택과 데이터 수집
10분 간격으로 24시간 측정합니다.

- 실제 장비는 `time.sleep(INTERVAL)`으로 측정 간격만큼 기다립니다.
- 가상 센서는 기다리지 않고 시간을 건너뛰므로, 몇 시간·며칠 분량의 데이터도 바로 만들어집니다.

In [ ]:
USE_REAL_SENSOR = False      # ← 라즈베리파이에 센서를 연결했다면 True 로 변경
N_SAMPLES = 144               # 측정 횟수
INTERVAL = 600               # 측정 간격(초)

sensor = RealSEN0189() if USE_REAL_SENSOR else VirtualSEN0189(dt=INTERVAL)

records = []
for i in range(N_SAMPLES):
    value = sensor.read()
    records.append({"time_s": round(i * INTERVAL, 4), **value})
    if USE_REAL_SENSOR:
        time.sleep(INTERVAL)

df = pd.DataFrame(records)
print(f"{len(df)}개 측정 완료 (총 {N_SAMPLES * INTERVAL / 60:.1f}분 분량)")
df.head()

In [ ]:
df.describe().round(3)

## 7. 판정 – 이벤트 감지
수조 관리 기준(교육용)으로 **5 NTU 초과 시 경보**, 50 NTU 초과 시 **원수 유입 차단**으로 판정합니다.

In [ ]:
df["hour"] = df["time_s"] / 3600
df["state"] = np.where(df["ntu"] > 50, "유입 차단", np.where(df["ntu"] > 5, "경보", "정상"))
print("상태별 시간(분):", (df["state"].value_counts() * INTERVAL / 60).to_dict())
alarm = df[df["state"] != "정상"]
if len(alarm):
    print(f"오염 감지 시작: {alarm['hour'].iloc[0]:.1f}시")
    ok = df[(df.hour > df.loc[df.ntu.idxmax(), 'hour']) & (df.ntu <= 5)]
    if len(ok): print(f"정상 복귀(5 NTU 이하): {ok['hour'].iloc[0]:.1f}시")
print(f"최고 탁도 {df['ntu'].max():.0f} NTU ({df.loc[df['ntu'].idxmax(), 'hour']:.1f}시)")
print(f"\n참고: 2 NTU ↔ {VirtualSEN0189.ntu_to_voltage(2):.4f}V, 120 NTU ↔ {VirtualSEN0189.ntu_to_voltage(120):.4f}V"
      " → 전압 차이가 매우 작아 고분해능 ADC가 필요합니다.")

## 8. 시각화

In [ ]:
fig, ax = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
ax[0].plot(df["hour"], df["ntu"], "k-", lw=1.5); ax[0].set_ylabel("탁도 (NTU)")
ax[0].axhline(5, color="gray", ls="--", label="경보 5 NTU"); ax[0].axhline(50, color="gray", ls=":", label="차단 50 NTU")
ax[0].legend(); ax[0].grid(alpha=0.3); ax[0].set_title("SEN0189 수질 탁도")
ax[1].plot(df["hour"], df["v_sensor"], "k-", lw=1.2); ax[1].set_ylabel("센서 전압 (V)")
ax[1].set_xlabel("시각 (시)"); ax[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 9. 저장과 IoT 전송
측정값은 CSV 파일로 저장하고, IoT 서버로 보낼 때는 보통 **JSON** 형식의 메시지를 **MQTT** 프로토콜로 전송합니다.
(Colab에서는 왼쪽 📁 메뉴에서 저장된 CSV를 내려받을 수 있습니다.)

In [ ]:
csv_name = "수질탁도센서_data.csv"
df.to_csv(csv_name, index=False, encoding="utf-8-sig")
print("CSV 저장:", csv_name)

def to_jsonable(v):
    if isinstance(v, (bool, np.bool_)): return bool(v)
    if isinstance(v, (int, np.integer)): return int(v)
    if isinstance(v, (float, np.floating)): return None if np.isnan(v) else round(float(v), 3)
    return str(v)

last = df.iloc[-1].to_dict()
payload = json.dumps({"device": "rpi-01", "sensor": "DFRobot SEN0189",
                      "time": datetime.now().isoformat(timespec="seconds"),
                      "values": {k: to_jsonable(v) for k, v in last.items()}}, ensure_ascii=False)
print("IoT 서버로 보낼 메시지(JSON):")
print(payload)

# ▼ 실제로 MQTT 브로커에 보내려면 (pip install paho-mqtt) 주석을 해제하세요.
# import paho.mqtt.client as mqtt
# client = mqtt.Client()
# client.connect("broker.hivemq.com", 1883)          # 공개 테스트 브로커
# client.publish("myclass/rpi-01/turbidity", payload)
# client.disconnect()

## 직접 해 보기
- MCP3008(10비트, 1단계 ≈ 4.8mV)을 썼다면 3 NTU와 20 NTU를 구분할 수 있을지 전압 차이로 계산해 보세요.
- 유입 차단 상태일 때 원수 밸브(GPIO 릴레이)를 닫는 코드를 생각해 보세요.
- 탁도가 다시 5 NTU 이하로 내려간 시각을 찾아 '정상 복귀' 메시지를 출력해 보세요.